# Dated payments: from a caplet to an amortizing note

A payoff script describes a contract as a list of dated events. Each event
can observe market rates and schedule payments. This notebook first writes a
caplet as a script. It then scripts the coupon and principal payments of a
six-year amortizing note that pays semiannually. Both use the illustrative
USD SOFR market dated 24 February 2026 and a Gaussian short-rate model fitted
to caplet prices.

The note's fixed coupon accrues on an Actual/365 basis and its floating
coupons on Actual/360. Dates are not adjusted for holidays. We value the
note's payments without the issuer's calls. Two checks follow:

- with zero volatility, the scripted note should match the library's built-in
  bond pricer;
- running the caplet with several random seeds shows the Monte Carlo
  sampling error.

[Setup](../README.md) · [Callable note](../valuation/bonds/stepped_coupon_bond.ipynb).


In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market, load_pricers
from stepped_cashflows import underlying_script

market = load_market()
ref = market.quotes.reference_date
pricers = load_pricers("bond_tree.json")


## Observing a rate and scheduling a payment

Each event runs on its date. In the caplet script:

- `RateIndex` observes SOFR for the caplet's accrual period;
- `cvg` computes the accrual fraction;
- `pays` adds the discounted payoff to the variable `note`.

The variables `rate` and `accrual` keep these intermediate values so you can
inspect them.

The rate model comes from the pricing context's configuration.
`result_variable` names the variable that is the contract's value, here
`note`. `evaluate_script` returns the average over paths of every script
variable, not just the result.


In [2]:
fixing = ref + "1Y"
payment = fixing + "3M"
scripted_caplet = qs.ScriptedProduct.from_dict(
    {
        "name": "SCRIPTED-CAPLET",
        "currency": qs.Currency.USD,
        "result_variable": "note",
        "events": [
            {
                "date": fixing,
                "code": (
                    f'note = 0; rate = RateIndex("SOFR", "{fixing}", "{payment}"); '
                    f'accrual = cvg("{fixing}", "{payment}", "Actual360"); '
                    f'note pays 1000000 * max(rate - 0.045, 0) * accrual on "{payment}";'
                ),
            }
        ],
    }
)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
    pricer_configurations=pricers,
)

with context:
    values = context.evaluate_script(scripted_caplet, n_paths=2048, seed=42)

pd.Series(values)


accrual       0.247222
note       1137.669549
rate          0.042082
dtype: float64

For scripts, the configured Hull–White model is converted to an equivalent
linear Gaussian model (LGM). Its volatility schedule is fitted to ATM caplet
prices at 3M, 6M, 1Y, 2Y, 3Y and 5Y. The last volatility segment is extended
past five years. If you change the market inputs, check that the ATM strikes
stay inside the surface's quoted strike range of 3.5% to 5.5%.

The scripted caplet is valued with this Gaussian model by Monte Carlo
simulation, so its value has sampling error. It need not equal a Black price:
the difference also depends on which caplets the model was fitted to and on
the option's strike.

## The stepped note's underlying payments

The note below has the coupon and principal schedule of the
[callable-bond notebook](../valuation/bonds/stepped_coupon_bond.ipynb), with
the calls removed. The [helper module](stepped_cashflows.py) turns this
semiannual schedule into dated script events. Read it next to the contract
if you want to change the payment rules.


In [3]:
underlying = qs.SteppedCouponBond(
    identifier="STEPPED-UNDERLYING",
    start_date=ref,
    maturity_date=ref + "6Y",
    notional=1_000_000,
    currency=qs.Currency.USD,
    payment_frequency=qs.Frequency.Semiannual,
    coupon_day_counter=qs.DayCounter.Actual365,
    coupons=[
        qs.FixedRateCoupon(rate=qs.InterestRate(rate=0.045)),
        qs.OptionEmbeddedCoupon(
            index=qs.MarketIndex.SOFR,
            payoff=(qs.PayoffOps.Index + 0.006).max(0.0).min(0.06),
        ),
    ],
    notional_percentages=[1.0, 0.90, 0.75],
    call_schedule=[],
)
script = underlying_script(underlying)
pd.DataFrame(script.to_dict()["events"]).head(3)


,date,code
0,2026-02-24,"bond = 0; accrual = cvg(""2026-02-24"", ""2026-08..."
1,2026-08-24,"accrual = cvg(""2026-08-24"", ""2027-02-24"", ""Act..."
2,2027-02-24,"accrual = cvg(""2027-02-24"", ""2027-08-24"", ""Act..."


The first event sets `bond` to zero and schedules the fixed coupon and the
first principal reduction. Each later event observes SOFR, adds the spread,
applies the floor and cap, and schedules the payments. The last event repays
the remaining principal. Fixed coupons use Actual/365 and floating coupons
use Actual/360.

We value this noncallable note at zero spread.


In [4]:
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
    pricer_configurations=pricers,
)

with context:
    stochastic = context.evaluate_script(script, n_paths=4096, seed=42)
print(f"Calibrated scripted underlying PV: USD {stochastic['bond']:,.2f}")


Calibrated scripted underlying PV: USD 1,010,885.99


## Checking the payment representation

To compare the built-in bond pricer with the script, we set volatility and
the Z-spread to zero. Rates are then certain, so one path is enough. The two
values should agree up to the tree's discretization error for floating
coupons, which we expect to be below 0.05% of the built-in PV.

The other examples in this notebook keep the calibrated model.


In [5]:
deterministic_spec = pricers[0].to_dict()
deterministic_spec["models"][0]["model"]["HullWhite"]["parameter_source"] = {
    "Fixed": {"sigma": 0.0}
}
deterministic_pricer = qs.PricerConfiguration.from_dict(deterministic_spec)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    pricer_configurations=[deterministic_pricer],
)

with context:
    native = context.evaluate(
        qs.Trade(underlying, ref, pricing_anchor=qs.BondPricingAnchor.z_spread(0.0)),
        [qs.Request.Value],
    )
    scripted = context.evaluate_script(script, n_paths=1, seed=42)

pd.Series(
    {
        "native_pv_usd": native.price,
        "scripted_pv_usd": scripted["bond"],
        "difference_usd": native.price - scripted["bond"],
    }
)


native_pv_usd      1.020546e+06
scripted_pv_usd    1.020546e+06
difference_usd     0.000000e+00
dtype: float64

## Sampling variation

The zero-volatility comparison checks the payment terms and amortization
with the same discounting on both sides. Next we value the scripted caplet
with several seeds to see how much the estimate varies. Seed 42 should give
the same value as the first run.

The result variable may include a control-variate adjustment, a technique
that reduces Monte Carlo noise. The raw discounted payments are not adjusted,
so with a finite number of paths their sum can differ from the result
variable. Variables such as `rate` are averages of observed values, while
`note` is the sum of the payments.


In [6]:
with qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
    pricer_configurations=pricers,
) as context:
    estimates = []
    for seed in [41, 42, 43, 44, 45]:
        estimate = context.evaluate_script(scripted_caplet, n_paths=2048, seed=seed)
        estimates.append({"seed": seed, "caplet_pv_usd": estimate["note"]})

estimates = pd.DataFrame(estimates)
print(
    f"Deterministic representation difference: USD {native.price - scripted['bond']:.6f}."
)
print(
    f"Across-seed SD of 2,048-path caplet estimates: USD {estimates['caplet_pv_usd'].std():.4f}."
)
estimates


Deterministic representation difference: USD 0.000000.
Across-seed SD of 2,048-path caplet estimates: USD 3.4266.


,seed,caplet_pv_usd
0,41,1136.335996
1,42,1137.669549
2,43,1129.344025
3,44,1137.054234
4,45,1136.635810


The standard deviation across seeds shows how much the 2,048-path estimates
vary. Run more paths to see whether it shrinks as expected. The note check
covers the payments only. To include the issuer's calls, use the
callable-bond pricer.


In Python, `evaluate_script` builds the rate models and returns the averaged
script variables as numbers. It does not return paths, cashflow tables or
sensitivity tables, and it does not support FX or equity factors or custom
correlations. When you compare a script with another representation of the
same contract, use the same terms, spread, model and number of paths.